In [ ]:
import pandas as pd
import numpy as np
from google.colab import sheets

# -------------------------------------------------------------------
# 1. carregar os dados
# -------------------------------------------------------------------

df = pd.read_csv("livros.csv", sep=";")

# -------------------------------------------------------------------
# 2. substituir 0 por NaN (para que fillna funcione)
# -------------------------------------------------------------------

df.replace(0, np.nan, inplace=True)

# -------------------------------------------------------------------
# 3. tratar valores nulos
# -------------------------------------------------------------------
# preencher os anos com a mediana 2012

df["ano"] = df["ano"].fillna(df["ano"].median()).astype("int32")

# preencher os autores (ou falta deles) com "Sem Crédito"

df["autor"] = df["autor"].fillna("Sem Crédito").astype("object")

# remover linhas com páginas inválidas depois

# -------------------------------------------------------------------
# 4. criar coluna de década com base no ano
# -------------------------------------------------------------------

df["decada"] = (df["ano"] // 10) * 10

# -------------------------------------------------------------------
# 5. remover linhas com páginas inválidas (NaN ou <= 0)
# -------------------------------------------------------------------
df_limpo = df[df["paginas"] > 0].copy()

# -------------------------------------------------------------------
# 6. classificar livros por faixa de páginas
#    - baixo: até 150 páginas
#    - médio: 151 a 350 páginas
#    - longo: mais de 350 páginas
# -------------------------------------------------------------------

def classificar_faixa(paginas):
    if paginas > 350:
        return "Longo"
    elif paginas > 150:
        return "Médio"
    else:
        return "Baixo"

df_limpo["faixa_paginas"] = df_limpo["paginas"].apply(classificar_faixa)

# -------------------------------------------------------------------
# 7. visualização interativa
# -------------------------------------------------------------------
sheet = sheets.InteractiveSheet(df=df_limpo)

# -------------------------------------------------------------------
# 8. análises exploratórias via solicitação
# -------------------------------------------------------------------
# quantidade de livros por ano
print(df["ano"].value_counts().sort_index())

# quick view
print(df.head())
print(df.describe())

# verificação de valores nulos
print(df.isna().sum())
print(df.info())

# mostra os registros com páginas nulas em df (mas não em df_limpo)
print(df[df["paginas"].isna()])

# média de páginas por década (usando df_limpo)
print(df_limpo.groupby("decada")["paginas"].mean())

# 10 autores com mais livros
print(df_limpo["autor"].value_counts().head(10))

# contagem de faixas de páginas para livros de 2010 em diante
print(df_limpo[df_limpo["decada"] > 2010]["faixa_paginas"].value_counts())

# -------------------------------------------------------------------
# 9. exportar para Excel
# -------------------------------------------------------------------
df_limpo.to_excel("livros_analisados.xlsx", index=False)